In [ ]:
import pandas as pd

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [17]:
print(train['text'].apply(lambda x: len(x.split())).describe())

count    120000.000000
mean         37.847450
std          10.085245
min           8.000000
25%          32.000000
50%          37.000000
75%          43.000000
max         177.000000
Name: text, dtype: float64


In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="textattack/bert-base-uncased-ag-news",
    device=0  # GPU; use -1 for CPU
)

all_preds_raw = classifier(
    test['text'].tolist(),
    truncation=True,
    max_length=128,
    batch_size=64       # lower to 32 if you get OOM
)

label_map = {"LABEL_0": 0, "LABEL_1": 1, "LABEL_2": 2, "LABEL_3": 3}
predicted_labels = [label_map[p['label']] for p in all_preds_raw]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 39180.89it/s]


In [ ]:
rows = []
for id, pred in zip(test['id'], predicted_labels):
    rows.append({
        'id': id,
        'label': pred
    })
sub = pd.DataFrame(rows)
sub.to_csv('submission.csv', index=False)
# .951 accuracy